# Classifying BBQ product reviews with Jev

A fictional barbecue retailer: customers, their orders over time, and the reviews they chose to write, generated deterministically (see [`docs/data-model.md`](../docs/data-model.md)). Every review sentence is then classified by [TypeSafe AI's Jev](https://docs.typesafe.ai/introduction), a *System One* model that answers typed questions (`Choice`, `Score`, `Noul`) with probabilities and a confidence, instead of generating text to be parsed.

1. Generate customers, orders and reviews, and write the review text.
2. Load the reviews, each with what we knew about the customer **at the moment they wrote it**.
3. Break each review into sentences.
4. Ask Jev about every sentence, with the whole review as context.
5. Roll the answers up into things a product or support team can act on.

All the logic lives in `src/`; this notebook only runs it. `TYPESAFE_API_KEY` (and, for generated text, the `AZURE_FOUNDRY_*` settings) are read from `.env`.

In [1]:
from datetime import datetime
from pathlib import Path

import polars as pl
from dotenv import load_dotenv

from jev_classifier import FRUSTRATION_LEVELS, KEY, PROBLEM_CATEGORIES, QUESTION_SET_VERSION, build_questions, classify_sentences, transcript
from retail_generator import add_customers, init_dataset, status
from retail_model import DEFAULT_DIR, read_dataset
from review_insights import cross_mentions, flagged, frustration_by_rating, needs_review, problems_by_product, review_rollup
from review_wrangler import load_reviews, product_catalogue, split_sentences
from review_writer import FoundryReviewWriter, TemplateReviewWriter

load_dotenv(Path.cwd().parent / ".env")
OUTPUT = Path.cwd().parent / "data" / "output"
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_cols(16);

# The dataset: change these to scale it. The same seed always gives the same data.
CUSTOMERS = 1000
SEED = 42
AS_OF = datetime(2026, 6, 30)
# Who writes review text: "foundry" (Azure AI Foundry, the intended source) or "template"
# (hand-written reviews reused offline; English only, and labelled as such).
WRITER = "template"

## 1. Generate the dataset

Customers come from Faker, one locale per country. Orders follow `OrderPatterns`: seasonality flipped for the Southern Hemisphere, rare grills, accessories that come with a new grill, fuel matched to the grill and restocked on a cycle. Each purchase has a hidden satisfaction, and people mostly review when they are delighted or unhappy. Review text is written from a brief per review and cached by prompt.

The dataset grows in batches, as the `generate-data` command does (`uv run generate-data --help`). Asking for a `total` is idempotent, so rerunning this notebook reuses what is there. To scale up, raise `CUSTOMERS`, or move time on with `generate-data advance --to <date>`. Each batch reports what it will cost.

In [2]:
init_dataset(DEFAULT_DIR, seed=SEED, as_of=AS_OF)
writer = FoundryReviewWriter.from_env() if WRITER == "foundry" else TemplateReviewWriter()
report = await add_customers(DEFAULT_DIR, total=CUSTOMERS, writer=writer)
await writer.aclose()
print("\n".join(report.lines() + [""] + status()))

1511 review text(s) to write with template:product_reviews.json, 0 cached.
Wrote 1511 in 0.0s; 0 failed.


Wrote batch 1 (add-customers): 1,000 customers, 8,441 orders, 12,706 order_lines, 1,511 reviews, 1,511 review_truth, 1,511 review_texts
1,511 new reviews, up to 6,803 sentences
Azure AI Foundry text: 392,860 input + 190,484 output tokens, set prices to cost it, ~7.9 min
Jev classification:    18,708,250 input tokens, $0.79, ~4.3 min (upper bound: cached sentences are free)

Dataset at /workspaces/retail-review-knowledge-mining/data/generated: seed 42, 2023-01-01 to 2026-06-30, 1,000 customers, 1 batch(es)
     1  add-customers  2026-09-18T18:17:27  template:product_reviews.json    1,000 customers, 8,441 orders, 12,706 order_lines, 1,511 reviews, 1,511 review_truth, 1,511 review_texts


Reviews are a **biased sample** of purchases: only a minority of purchases are reviewed, and the stars lean to the extremes.

In [3]:
tables = read_dataset()
purchases = tables["order_lines"].height
reviewed = tables["reviews"].height
print(f"{reviewed:,} of {purchases:,} purchases reviewed ({reviewed / purchases:.1%})")
tables["reviews"]["rating"].value_counts().sort("rating").with_columns(share=(pl.col("count") / reviewed).round(3))

1,511 of 12,706 purchases reviewed (11.9%)


rating,count,share
i8,u32,f64
1,109,0.072
2,185,0.122
3,176,0.116
4,577,0.382
5,464,0.307


## 2. Load the reviews

One row per review, joined to its text, its product and **point-in-time customer features**: tenure, lifetime revenue, orders so far and age, all computed only from what happened before the review was written. `review_key` hashes product + text, so each distinct text is classified once.

In [4]:
reviews = load_reviews()
reviews_df = reviews.collect()
print(reviews_df.shape, "-", reviews_df["review_key"].n_unique(), "distinct texts")
reviews_df.head()

(1511, 19) - 150 distinct texts


review_id,review_key,customer_id,product_id,product_name,product_category,rating,reviewed_at,…,tenure_days,lifetime_revenue,order_count,days_since_last_order,previous_reviews,age,gender,country
str,str,str,str,str,str,i8,datetime[μs],…,f64,f64,u32,f64,u32,i32,str,str
"""R0000001010""","""54fa0951850e""","""C0000001""","""P014""","""SmokeRing Premium Lump Charcoal""","""consumables""",4,2023-10-30 14:48:03.730870,…,229.477118,496.8,2,5.071366,1,64,"""male""","""Costa Rica"""
"""R0000001011""","""654c28ed1f95""","""C0000001""","""P015""","""WoodChips Hickory Smoking Chips""","""consumables""",2,2023-10-30 09:38:59.730870,…,229.262488,496.8,2,4.856736,0,64,"""male""","""Costa Rica"""
"""R0000001140""","""dd742e24c348""","""C0000001""","""P008""","""GrillMaster Elite Tongs""","""accessories""",5,2025-10-26 21:18:26.615662,…,956.748218,1390.14,15,5.957014,2,66,"""male""","""Costa Rica"""
"""R0000001142""","""4b764b6fe747""","""C0000001""","""P016""","""FlameStarter Natural Fire Lighter""","""consumables""",5,2025-11-11 13:26:10.615662,…,972.420255,1390.14,15,21.629051,3,66,"""male""","""Costa Rica"""
"""R0000002000""","""fdf4094ab4d0""","""C0000002""","""P017""","""SeasonPro BBQ Rub Collection""","""consumables""",4,2023-11-28 05:21:27.375884,…,118.510231,36.2,1,15.828391,0,41,"""male""","""New Zealand"""


A customer's features move with time. Here is the customer with the most reviews: each review sees only the orders placed before it.

In [5]:
busiest = reviews_df.group_by("customer_id").len().sort("len", "customer_id", descending=[True, False])["customer_id"][0]
reviews_df.filter(pl.col("customer_id") == busiest).select(
    "reviewed_at", "product_name", "rating", pl.col("tenure_days").round(), "lifetime_revenue", "order_count", "previous_reviews"
)

reviewed_at,product_name,rating,tenure_days,lifetime_revenue,order_count,previous_reviews
datetime[μs],str,i8,f64,f64,u32,u32
2023-10-29 12:29:47.846,"""CleanBurn Pellets""",5,192.0,1343.23,5,0
2023-11-29 14:31:56.460288,"""CleanBurn Pellets""",5,223.0,1720.78,9,1
2024-02-27 14:19:13.116413,"""FlameStarter Natural Fire Lighter""",1,313.0,1887.33,11,2
2024-04-27 14:14:11.506055,"""SmokeRing Premium Lump Charcoal""",4,373.0,2091.68,14,3
2024-06-12 08:04:05.769885,"""SmokeRing Premium Lump Charcoal""",4,419.0,2176.16,15,4
2024-10-14 17:50:05.999772,"""CleanBurn Pellets""",5,543.0,2320.66,17,5
2024-11-14 11:23:33.406340,"""CleanBurn Pellets""",5,574.0,2454.83,18,6
2025-04-26 20:20:02.765150,"""CleanBurn Pellets""",5,737.0,2720.77,21,7
2026-01-22 02:10:42.680895,"""WoodChips Hickory Smoking Chips""",2,1007.0,3279.13,27,8


The **master product list**, the options Jev picks mentions from:

In [6]:
catalogue = product_catalogue()
catalogue

product_name,product_category
str,str
"""ChefsPride Stainless Steel Spatula Set""","""accessories"""
"""FlipMaster Long Handle Fork""","""accessories"""
"""GrillGuard Heat Resistant Gloves""","""accessories"""
"""GrillMaster Elite Tongs""","""accessories"""
"""HeatShield Premium Grill Cover""","""accessories"""
"""TempCheck Digital Thermometer""","""accessories"""
"""CleanBurn Pellets""","""consumables"""
"""FlameStarter Natural Fire Lighter""","""consumables"""
"""SeasonPro BBQ Rub Collection""","""consumables"""


## 3. Break reviews into sentences

In [7]:
sentences = split_sentences(reviews).collect()
print(f"{sentences.height} sentences, {sentences.select(KEY).n_unique()} distinct to classify")
sentences.select("review_id", "product_name", "sentence_index", "sentence_count", "sentence").head(12)

6530 sentences, 607 distinct to classify


review_id,product_name,sentence_index,sentence_count,sentence
str,str,u32,u32,str
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",0,4,"""High quality lump charcoal that lights easily and burns very hot."""
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",1,4,"""Perfect for getting a good sear on steaks."""
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",2,4,"""Minimal ash makes cleanup simple."""
"""R0000001010""","""SmokeRing Premium Lump Charcoal""",3,4,"""Burns a bit fast but the intense heat makes it worth it for high temperature cooking."""
"""R0000001011""","""WoodChips Hickory Smoking Chips""",0,3,"""Wood chips are too large and don't soak water well."""
"""R0000001011""","""WoodChips Hickory Smoking Chips""",1,3,"""Burn too fast and produce bitter smoke."""
"""R0000001011""","""WoodChips Hickory Smoking Chips""",2,3,"""Better options available for less money."""
"""R0000001140""","""GrillMaster Elite Tongs""",0,5,"""Perfect tongs!"""
"""R0000001140""","""GrillMaster Elite Tongs""",1,5,"""Length is ideal for safety and the spring action feels just right."""


## 4. Ask Jev about every sentence

Each call sends one sentence **plus the whole review** as JSON state, and asks 26 questions at once - Jev evaluates them in parallel, so extra questions barely cost latency.

| Question | Type | Why |
|---|---|---|
| `frustration` | Score, 5 levels | How frustrated is the customer? A continuous 0-4. |
| `problem_category` | Choice | Product quality, shipping, ease of use... or `none`. |
| `mentions__<product>` × 17 | Noul each | Which catalogue products the sentence refers to. One Noul per product, because a sentence can mention several. |
| `language` | Choice | Which common language it is written in. |
| `sentiment` | Choice | positive / negative / mixed / neutral. |
| `recommendation` | Choice | Recommends / warns others off / neither. |
| `churn_risk` | Noul | Returning it, refund, switching brand, won't buy again. |
| `safety_concern` | Noul | Burns, fire, gas, unsafe food - an escalation trigger. |
| `suggestion` | Noul | An explicit product improvement idea. |
| `competitor_mention` | Noul | Mentions a product outside our range. |

The star rating and demographics are deliberately **not** sent - see the sanity check below. Answers are cached in `data/output/`, so re-running only pays for new sentences.

In [8]:
questions = build_questions(catalogue.to_dicts())
print(len(questions), "questions per sentence; question set", QUESTION_SET_VERSION)
print("frustration rubric:", *[f"  {i}: {level}" for i, level in enumerate(FRUSTRATION_LEVELS)], sep="\n")

26 questions per sentence; question set 2026-09-18.3
frustration rubric:
  0: Not frustrated: positive, neutral or purely factual.
  1: Mild: a minor gripe or slight disappointment, said calmly.
  2: Clearly frustrated: annoyed or let down by a real problem.
  3: Very frustrated: angry, feels cheated, or the problem ruined the experience.
  4: Furious: hostile or emphatic language, demands a refund, or vows never to buy again.


In [9]:
answers = await classify_sentences(sentences, catalogue, cache_path=OUTPUT / "jev_sentence_answers.parquet")

answers.select(
    pl.col("jev_model").unique().alias("model"),
    pl.col("latency_ms").mean().round().alias("mean_latency_ms"),
    pl.col("input_tokens").sum().alias("input_tokens"),
    pl.col("error").is_not_null().sum().alias("errors"),
)

0 sentence(s) to classify, 693 cached.


model,mean_latency_ms,input_tokens,errors
str,f64,i64,u32
"""jev-1.13.0""",351.0,1925826,0


### Watch the wire

`transcript.log_to_stdout()` prints every exchange with Jev as JSON: the request body exactly as the SDK sent it (state, model alias, questions) and the response body exactly as the API returned it (the versioned model, every answer with its full probability distribution, token usage), plus the request ID and latency. The question set is printed in full once and elided after that - pass `full_questions=True` to see it every time. The API key travels in a header and is never printed.

The run above came from the cache, so this asks about two sentences afresh. To watch the whole run instead, call `transcript.log_to_stdout()` before it and delete `data/output/jev_sentence_answers.parquet`.

In [10]:
transcript.log_to_stdout()
await classify_sentences(sentences.head(2), catalogue)  # no cache_path, so these always call the API
transcript.silence()

2 sentence(s) to classify, 0 cached.


{
  "at": "2026-09-18T18:17:28.421+00:00",
  "sentence": {"review_key": "54fa0951850e", "sentence_index": 0},
  "request_id": "req_01a0b5bcdbc87a3fa79a1119541b74bc",
  "latency_ms": 654,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "High quality lump charcoal that lights easily and burns very hot.",
      "sentence_position": "1 of 4",
      "review": {
        "product_reviewed": "SmokeRing Premium Lump Charcoal",
        "product_category": "consumables",
        "full_text": "High quality lump charcoal that lights easily and burns very hot. Perfect for getting a good sear on steaks. Minimal ash makes cleanup simple. Burns a bit fast but the intense heat makes it worth it for high temperature cooking."
      }
    },
    "model": "jev-latest",
    "questions": {
      "frustra

{
  "at": "2026-09-18T18:17:28.481+00:00",
  "sentence": {"review_key": "54fa0951850e", "sentence_index": 1},
  "request_id": "req_01a0b5bcdbe572559b0d48a828a7e685",
  "latency_ms": 699,
  "sent": {
    "state": {
      "task": "Classify one sentence from a customer review of a barbecue product.",
      "note": "Judge what the sentence itself says. The full review is context for resolving what 'it' or 'this' refers to.",
      "sentence": "Perfect for getting a good sear on steaks.",
      "sentence_position": "2 of 4",
      "review": {
        "product_reviewed": "SmokeRing Premium Lump Charcoal",
        "product_category": "consumables",
        "full_text": "High quality lump charcoal that lights easily and burns very hot. Perfect for getting a good sear on steaks. Minimal ash makes cleanup simple. Burns a bit fast but the intense heat makes it worth it for high temperature cooking."
      }
    },
    "model": "jev-latest",
    "questions": "<elided: the same 26 questions as the 

Classified 2 in 0.7s; 0 failed.


Join the answers back onto every sentence (duplicated reviews pick up the same answers) and save the flat result.

In [11]:
classified = sentences.join(answers, on=KEY, how="left")
classified.drop("answers_json").write_parquet(OUTPUT / "sentences_classified.parquet")
classified.select(
    "product_name", "sentence", pl.col("frustration").round(2), "problem_category", "sentiment", "products_mentioned", "language"
).head(15)

product_name,sentence,frustration,problem_category,sentiment,products_mentioned,language
str,str,f64,str,str,list[str],str
"""SmokeRing Premium Lump Charcoal""","""High quality lump charcoal that lights easily and burns very hot.""",0.0,"""none""","""positive""","[""SmokeRing Premium Lump Charcoal""]","""english"""
"""SmokeRing Premium Lump Charcoal""","""Perfect for getting a good sear on steaks.""",0.0,"""none""","""positive""","[""SmokeRing Premium Lump Charcoal""]","""english"""
"""SmokeRing Premium Lump Charcoal""","""Minimal ash makes cleanup simple.""",0.0,"""none""","""positive""","[""SmokeRing Premium Lump Charcoal""]","""english"""
"""SmokeRing Premium Lump Charcoal""","""Burns a bit fast but the intense heat makes it worth it for high temperature cooking.""",0.64,"""performance""","""mixed""","[""SmokeRing Premium Lump Charcoal""]","""english"""
"""WoodChips Hickory Smoking Chips""","""Wood chips are too large and don't soak water well.""",1.79,"""performance""","""negative""","[""WoodChips Hickory Smoking Chips""]","""english"""
"""WoodChips Hickory Smoking Chips""","""Burn too fast and produce bitter smoke.""",1.95,"""performance""","""negative""","[""WoodChips Hickory Smoking Chips""]","""english"""
"""WoodChips Hickory Smoking Chips""","""Better options available for less money.""",1.46,"""price_value""","""negative""","[""WoodChips Hickory Smoking Chips""]","""english"""
"""GrillMaster Elite Tongs""","""Perfect tongs!""",0.0,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""english"""
"""GrillMaster Elite Tongs""","""Length is ideal for safety and the spring action feels just right.""",0.0,"""none""","""positive""","[""GrillMaster Elite Tongs""]","""english"""


## 5. What the answers say

### Sanity check: does frustration track the star rating?

Jev never saw the rating, so if frustration falls as stars rise, it is reading the text rather than the number.

In [12]:
by_review = review_rollup(classified.lazy())
frustration_by_rating(by_review).collect()

rating,reviews,peak_frustration,mean_frustration
i8,u32,f64,f64
1,109,2.93,2.5
2,185,1.73,1.55
3,176,1.07,0.83
4,577,0.8,0.22
5,464,0.05,0.01


### Which problems, for which products

In [13]:
problems = problems_by_product(classified.lazy()).collect()
problems.pivot("problem_category", index="product_name", values="sentences").fill_null(0).sort("product_name")

product_name,performance,price_value,cleaning_maintenance,build_quality,ease_of_use,shipping_delivery,other,general_dissatisfaction,size_capacity,customer_service,safety
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""BBQ Baron Pellet Smoker""",7,0,0,0,2,0,0,0,9,0,0
"""BackYard King Compact Grill""",42,16,6,53,0,4,0,34,0,25,0
"""ChefsPride Stainless Steel Spatula Set""",0,11,0,3,14,0,0,0,0,0,12
"""CleanBurn Pellets""",75,97,0,0,0,0,0,0,0,0,0
"""FireMaster Pro 3000 Gas Grill""",0,2,0,0,1,0,0,0,0,0,0
"""FlameForge Electric Indoor Grill""",30,3,0,0,0,0,0,3,0,0,0
"""FlameStarter Natural Fire Lighter""",87,0,0,0,0,0,42,5,0,0,18
"""FlipMaster Long Handle Fork""",0,14,5,16,25,0,0,0,9,0,0
"""GrillGuard Heat Resistant Gloves""",7,0,0,0,50,0,0,0,25,7,14


In [14]:
# The most frustrating problem areas, where at least 3 reviews raise them
problems.filter(pl.col("reviews") >= 3).sort("mean_frustration", descending=True).head(10)

product_name,problem_category,sentences,reviews,mean_frustration
str,str,u32,u32,f64
"""BackYard King Compact Grill""","""general_dissatisfaction""",34,22,3.4
"""FlameForge Electric Indoor Grill""","""general_dissatisfaction""",3,3,2.99
"""TurboGrill Portable Gas""","""general_dissatisfaction""",11,8,2.97
"""BackYard King Compact Grill""","""customer_service""",25,21,2.96
"""BackYard King Compact Grill""","""price_value""",16,16,2.9
"""GrillGuard Heat Resistant Gloves""","""safety""",14,7,2.88
"""GrillGuard Heat Resistant Gloves""","""performance""",7,7,2.74
"""FlameForge Electric Indoor Grill""","""price_value""",3,3,2.65
"""GrillGuard Heat Resistant Gloves""","""customer_service""",7,7,2.59


### Language

Template text is always English, so with `WRITER = "template"` this is a guard. With Azure AI Foundry, about a third of customers in non-English-speaking countries write in their own language, and this becomes a finding.

In [15]:
classified.group_by("language").agg(pl.len(), pl.col("language_confidence").min().alias("min_confidence"))

language,len,min_confidence
str,u32,f64
"""english""",6530,1.0


### Products mentioned in reviews of *other* products

Compatibility and bundling signals - e.g. a cover reviewed alongside the grill it fits.

In [16]:
cross_mentions(classified.lazy()).collect()

product_name,also_mentions,sentence
str,str,str
"""HeatShield Premium Grill Cover""","""FireMaster Pro 3000 Gas Grill""","""Solid grill cover that fits my FireMaster perfectly."""


### Escalations: safety concerns and churn risk

In [17]:
flagged(classified.lazy(), "safety_concern").collect()

product_name,sentence,safety_concern,occurrences,frustration
str,str,f64,u32,f64
"""GrillGuard Heat Resistant Gloves""","""Claimed to handle 932°F but my hands got burned picking up a moderately hot grate.""",0.97,7,2.52
"""TurboGrill Portable Gas""","""Gas connections leak.""",0.97,3,1.51
"""GrillGuard Heat Resistant Gloves""","""False advertising and dangerous.""",0.92,7,3.24
"""ChefsPride Stainless Steel Spatula Set""","""Handles get extremely hot and become uncomfortable to use.""",0.78,3,1.98
"""ChefsPride Stainless Steel Spatula Set""","""Only complaint is the handles get quite hot during longer cooking sessions.""",0.77,1,1.0
"""BackYard King Compact Grill""","""The temperature gauge is completely inaccurate, shows 300°F when it's clearly much hotter,…",0.76,4,2.99
"""TempCheck Digital Thermometer""","""Shows 200°F when meat is clearly still raw, then jumps to 250°F instantly.""",0.71,1,1.99
"""TurboGrill Portable Gas""","""Legs are wobbly and the whole thing feels like it might tip over.""",0.71,2,1.98
"""BackYard King Compact Grill""","""Temperature control doesn't work and the whole thing feels like it's about to collapse.""",0.7,4,2.78


In [18]:
flagged(classified.lazy(), "churn_risk").collect()

product_name,sentence,churn_risk,occurrences,frustration
str,str,f64,u32,f64
"""BackYard King Compact Grill""","""I'm returning it and will never buy from this company again.""",0.97,4,4.0
"""GrillMaster Elite Tongs""","""Going back to my old set.""",0.92,3,1.48
"""GrillGuard Heat Resistant Gloves""","""Returning immediately.""",0.92,7,2.59
"""GrillGuard Heat Resistant Gloves""","""Prefer my old leather gloves.""",0.84,4,0.93
"""WoodChips Hickory Smoking Chips""","""Find a different supplier.""",0.78,9,2.63
"""CleanBurn Pellets""","""Find a better brand.""",0.73,1,2.27
"""TempCheck Digital Thermometer""","""Waste of money - get a different brand.""",0.73,2,2.87
"""WoodChips Hickory Smoking Chips""","""Get your chips elsewhere.""",0.72,9,2.58
"""ChefsPride Stainless Steel Spatula Set""","""Save your money and buy quality tools elsewhere.""",0.67,3,2.73


### Free product ideas: explicit suggestions

In [19]:
flagged(classified.lazy(), "suggestion").collect()

product_name,sentence,suggestion,occurrences,frustration
str,str,f64,u32,f64
"""HeatShield Premium Grill Cover""","""Only wish it came in different colors besides basic black.""",0.98,2,0.99
"""FlipMaster Long Handle Fork""","""Only wish the handle had better grip texture.""",0.96,7,1.0
"""TempCheck Digital Thermometer""","""Only downside is the probe cord could be longer for larger grills.""",0.95,9,1.0
"""TempCheck Digital Thermometer""","""Only complaint is the probe cables could be longer for larger grills.""",0.95,9,1.0
"""BBQ Baron Pellet Smoker""","""Only complaint is the hopper could be larger for really long cooks.""",0.95,6,1.0
"""FlipMaster Long Handle Fork""","""Sturdy construction but the handle could be more comfortable for long cooking sessions.""",0.89,3,1.0
"""BBQ Baron Pellet Smoker""","""Hopper capacity could be larger for really long cooks but overall very satisfied with perf…",0.88,1,0.83
"""FlipMaster Long Handle Fork""","""Fork does the job but handle could be more comfortable.""",0.83,6,1.0
"""ChefsPride Stainless Steel Spatula Set""","""Handles could be more comfortable but overall good value.""",0.8,5,0.95


### Competitors and previous products

In [20]:
flagged(classified.lazy(), "competitor_mention").collect()

product_name,sentence,competitor_mention,occurrences,frustration
str,str,f64,u32,f64
"""GrillGuard Heat Resistant Gloves""","""Prefer my old leather gloves.""",0.94,4,0.93
"""GrillMaster Elite Tongs""","""Going back to my old set.""",0.92,3,1.48
"""GrillMaster Elite Tongs""","""After years of using cheap ones that broke or bent, these are a revelation.""",0.8,8,0.14
"""CleanBurn Pellets""","""Doesn't impart much smoke taste compared to other brands I've tried.""",0.68,12,1.01
"""TempCheck Digital Thermometer""","""Waste of money - get a different brand.""",0.62,2,2.87
"""GrillGuard Heat Resistant Gloves""","""Heat protection is good but dexterity suffers.""",0.62,4,1.05
"""SmokeRing Premium Lump Charcoal""","""Only downside is it burns a bit faster than some other brands so you go through it quicker…",0.61,33,1.0
"""GrillGuard Heat Resistant Gloves""","""Gloves are bulky and make it hard to grip smaller items.""",0.53,4,1.28
"""SmokeRing Premium Lump Charcoal""","""Burns clean but nothing special compared to other premium brands at this price point.""",0.52,32,1.02


### Low confidence: route to a person

Jev returns a confidence with every Choice. Where it could not separate the problem categories, that is a signal to send the sentence for human review rather than trust the label.

In [21]:
needs_review(classified.lazy(), min_confidence=0.6).collect()

product_name,sentence,problem_category,problem_category_confidence
str,str,str,f64
"""FlameStarter Natural Fire Lighter""","""Works but there are better natural options available.""","""other""",0.21
"""WoodChips Hickory Smoking Chips""","""Better chips available elsewhere.""","""performance""",0.23
"""TempCheck Digital Thermometer""","""App could use some improvements but overall very satisfied.""","""none""",0.3
"""BBQ Baron Pellet Smoker""","""Pellet smoker works as advertised but WiFi connectivity is unreliable.""","""performance""",0.3
"""GrillGuard Heat Resistant Gloves""","""Returning immediately.""","""customer_service""",0.31
"""WoodChips Hickory Smoking Chips""","""Wood chips are full of dirt and bark.""","""performance""",0.35
"""WoodChips Hickory Smoking Chips""","""Terrible wood chips full of dust and debris.""","""build_quality""",0.36
"""CleanBurn Pellets""","""Inconsistent sizes cause feeding problems in my smoker.""","""performance""",0.37
"""WoodChips Hickory Smoking Chips""","""Some pieces are too large while others are sawdust.""","""size_capacity""",0.38


### Most frustrated reviews

In [22]:
by_review.sort("peak_frustration", descending=True).select(
    "rating", "product_name", pl.col("peak_frustration").round(2), "problems", "churn_risk", "safety_concern", "review_text"
).unique("review_text", maintain_order=True).head(10).collect()

rating,product_name,peak_frustration,problems,churn_risk,safety_concern,review_text
i8,str,f64,list[str],bool,bool,str
1,"""BackYard King Compact Grill""",4.0,"[""build_quality"", ""cleaning_maintenance"", … ""shipping_delivery""]",true,true,"""Total waste of money! This grill arrived with multiple dents and scratches, clearly damage…"
1,"""BackYard King Compact Grill""",3.91,"[""build_quality"", ""general_dissatisfaction"", ""performance""]",false,false,"""Worst grill purchase ever made. Everything about this BackYard King is cheap and poorly de…"
1,"""BackYard King Compact Grill""",3.62,"[""customer_service"", ""general_dissatisfaction"", … ""price_value""]",false,false,"""Absolutely terrible grill with major design flaws. Temperature control doesn't work, heat …"
1,"""BackYard King Compact Grill""",3.59,"[""build_quality"", ""cleaning_maintenance"", … ""price_value""]",false,false,"""Worst grill ever made. Everything about it is cheap and poorly designed. The temperature k…"
1,"""TurboGrill Portable Gas""",3.51,"[""build_quality"", ""general_dissatisfaction"", ""safety""]",false,true,"""Portable grill is a disaster. Ignition system failed on the second use. Gas connections le…"
1,"""BackYard King Compact Grill""",3.36,"[""build_quality"", ""general_dissatisfaction"", ""performance""]",false,true,"""This grill is a complete disaster. Poor quality materials, terrible design, and non-existe…"
1,"""GrillGuard Heat Resistant Gloves""",3.24,"[""customer_service"", ""performance"", ""safety""]",true,true,"""Heat resistant gloves are a joke. Claimed to handle 932°F but my hands got burned picking …"
1,"""BackYard King Compact Grill""",3.07,"[""build_quality"", ""customer_service"", ""performance""]",true,true,"""Terrible quality control. Grill arrived with uneven legs so it wobbles constantly. Burner …"
1,"""FlipMaster Long Handle Fork""",3.04,"[""ease_of_use"", ""price_value"", ""size_capacity""]",false,false,"""Fork is poorly designed with inadequate handle length. Tines are too close together for ef…"
